In [51]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import r2_score

In [52]:
# ---------- Data ----------
df = fetch_california_housing()
X = df.data
y = df.target

In [53]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,random_state=42)

In [54]:
# ---------- Baseline: Plain Linear Regression ----------
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
r2_lr = r2_score(y_test, y_pred_lr)
print("Linear Regression accuracy : ", r2_lr )

Linear Regression accuracy :  0.5757877060324521


In [55]:
# ---------- L2: Ridge Regression ----------
ridge = Ridge(alpha= 1.0)  #alpha = lamda 
ridge.fit(X_train, y_train)
y_pred_ridge = ridge.predict(X_test)
r2_ridge = r2_score(y_test, y_pred_ridge)
print("Ridge Regression accuracy : ", r2_ridge )
print("Ridge coefficients:", np.round(ridge.coef_, 3))

Ridge Regression accuracy :  0.5758549611440138
Ridge coefficients: [ 0.449  0.01  -0.123  0.781 -0.    -0.004 -0.42  -0.434]


In [56]:
# ---------- L1: Lasso Regression ----------
lasso = Lasso(alpha= 0.09)
lasso.fit(X_train, y_train)
y_pred_lasso = lasso.predict(X_test)
r2_lasso = r2_score(y_test, y_pred_lasso)
print("Lasso Regression accuracy : ", r2_lasso)
print("Lasso coefficients:", np.round(lasso.coef_, 3))  # kuch exactly 0.0 honge

Lasso Regression accuracy :  0.5412362625973306
Lasso coefficients: [ 0.392  0.015 -0.     0.     0.    -0.003 -0.147 -0.134]


In [57]:
# ---------- Elastic Net (L1 + L2) ----------
elastic = ElasticNet(alpha=1.0, l1_ratio=0.5)   # l1_ratio: 0=pure Ridge, 1=pure Lasso
elastic.fit(X_train, y_train)
y_pred_elastic = elastic.predict(X_test)
r2_elastic = r2_score(y_test, y_pred_elastic)
print("ElasticNet R2 (alpha=1.0, l1_ratio=0.5):", r2_elastic)
print("ElasticNet coefficients:", np.round(elastic.coef_, 3))

ElasticNet R2 (alpha=1.0, l1_ratio=0.5): 0.41655189098028234
ElasticNet coefficients: [ 0.255  0.011  0.    -0.     0.    -0.    -0.    -0.   ]


In [59]:
#----------------------Manual Alpha Tuning----------------------------

alphas = [1.0, 0.5, 0.1, 0.01, 0.001]

print("---- Lasso alpha tuning ----")
for a in alphas:
    model = Lasso(alpha=a, max_iter=10000)
    model.fit(X_train, y_train)
    score = r2_score(y_test, model.predict(X_test))
    print(f"alpha={a} -> R2={score:.4f}")

print("---- Ridge alpha tuning ----")
for a in alphas:
    model = Ridge(alpha=a)
    model.fit(X_train, y_train)
    score = r2_score(y_test, model.predict(X_test))
    print(f"alpha={a} -> R2={score:.4f}")

print("---- ElasticNet alpha tuning ----")
for a in alphas:
    model = ElasticNet(alpha=a, l1_ratio=0.5, max_iter=10000)
    model.fit(X_train, y_train)
    score = r2_score(y_test, model.predict(X_test))
    print(f"alpha={a} -> R2={score:.4f}")

---- Lasso alpha tuning ----
alpha=1.0 -> R2=0.2842
alpha=0.5 -> R2=0.4457
alpha=0.1 -> R2=0.5318
alpha=0.01 -> R2=0.5845
alpha=0.001 -> R2=0.5773
---- Ridge alpha tuning ----
alpha=1.0 -> R2=0.5759
alpha=0.5 -> R2=0.5758
alpha=0.1 -> R2=0.5758
alpha=0.01 -> R2=0.5758
alpha=0.001 -> R2=0.5758
---- ElasticNet alpha tuning ----
alpha=1.0 -> R2=0.4166
alpha=0.5 -> R2=0.4758
alpha=0.1 -> R2=0.5627
alpha=0.01 -> R2=0.5836
alpha=0.001 -> R2=0.5771


In [64]:
#-------------------Automated alpha tuning using GridSearchCV----------------

from sklearn.model_selection import GridSearchCV   #GridSearchCV is used to find the best parameter(here alpha) automatically
params =  [0.001, 0.01, 0.1, 0.5, 1.0] #define the parameter grid
# Lasso
lasso_params = {'alpha': params}  #alpha is the name of the parameter and for tht try the values given in the params 
lasso_grid = GridSearchCV(   #Create a GridSearchCV object that takes 5 parameters
    Lasso(max_iter=10000),  #empty lasso base model 
    lasso_params,   #the dict of key alpha and its params 
    cv=5,  #5 fold cross validation
    scoring='r2')   #what evalution metric to use 
lasso_grid.fit(X_train, y_train)
#when you do fit this happens internally 

# for alpha in [0.001, 0.01, 0.1, 0.5, 1.0]:      # har alpha value ke liye
#     for fold in range(5):                        # 5-fold cross-validation
#         model = Lasso(alpha=alpha, max_iter=10000)
#         model.fit(4_folds_training_data)
#         score = r2_score(1_fold_validation_data, model.predict(...))
#     average_score_for_this_alpha = mean(5 scores)



print("Best Lasso alpha:", lasso_grid.best_params_, "-> R2:", lasso_grid.best_score_)

# Ridge
ridge_params = {'alpha': params}
ridge_grid = GridSearchCV(Ridge(), ridge_params, cv=5, scoring='r2')
ridge_grid.fit(X_train, y_train)
print("Best Ridge alpha:", ridge_grid.best_params_, "-> R2:", ridge_grid.best_score_)

# ElasticNet (dono hyperparameters ek saath tune)
elastic_params = {
    'alpha': params,
    'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
}
elastic_grid = GridSearchCV(ElasticNet(max_iter=10000), elastic_params, cv=5, scoring='r2')
elastic_grid.fit(X_train, y_train)
print("Best ElasticNet params:", elastic_grid.best_params_, "-> R2:", elastic_grid.best_score_)

Best Lasso alpha: {'alpha': 0.001} -> R2: 0.6114760503819732
Best Ridge alpha: {'alpha': 1.0} -> R2: 0.611485685843425
Best ElasticNet params: {'alpha': 0.001, 'l1_ratio': 0.1} -> R2: 0.6114884728071756
